In [1]:
!pip install pandas

In [2]:
!pip install scikit-learn

In [3]:
!pip install gTTs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.1
    Uninstalling click-8.4.1:
      Successfully uninstalled click-8.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.27.2 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
typer 0.25.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.18.0 requires click>=8.4.0, but you have click 8.1.8 which is incompatible.


In [4]:
!pip install fuzzywuzzy

In [5]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.4.1 which is incompatible.


In [8]:
import pandas as pd

recipe = pd.read_csv("/content/recipes_ingredients.csv",on_bad_lines = "skip", engine = "python")

In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
recipe["text"] = recipe["Name"].fillna("") + " " + recipe["Ingredients"].fillna("") + " " + recipe["Steps"].fillna("")

In [12]:
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(recipe["text"])

In [13]:
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(recipe["text"])

def search_recipes(query, top_n=3):
    query_vec = vectorizer.transform([query])
    similarity = cosine_similarity(query_vec, tfidf_matrix).flatten()
    indices = similarity.argsort()[::-1][:top_n]

    results = []
    for i in indices:
        results.append({
            "Name": recipe.iloc[i]["Name"],
            "Description": recipe.iloc[i]["Description"],
            "Ingredients": recipe.iloc[i]["Ingredients"],
            "Ingredients_raw": recipe.iloc[i]["Ingredients_raw"],
            "Steps": recipe.iloc[i]["Steps"],
            "Servings": recipe.iloc[i]["Servings"],
            "Serving_size": recipe.iloc[i]["Serving_size"],
            "Tags": recipe.iloc[i]["Tags"],
            "score": similarity[i]
        })
    return results
print("Smart Cooking Assistant Ready! (type 'exit' to quit)")

while True:
    query = input("\nWhat do you want to cook? ")
    if query.lower() == "exit":
        break

    results = search_recipes(query)
    if results:
        print("\nTop Recipes:")
        for r in results:
            print(f"\n {r['Name']} (score: {r['score']:.2f})")
            print("Description:",r["Description"][:200],"...")
            print("Ingredients:", r["Ingredients"])
            print("Ingredients_raw:", r["Ingredients_raw"])
            print("Steps:", r["Steps"][:200], "...")
            print("Servings",r["Servings"])
            print("Serving_size",r["Serving_size"])
            print("Tags", r["Tags"][:200], "...")

    else:
        print("Sorry, no recipe found.")


Smart Cooking Assistant Ready! (type 'exit' to quit)

What do you want to cook? Cherry Streusel Cobbler

Top Recipes:

 Vanilla Polenta Oatmeal Streusel (score: 0.50)
Description: Streusel is a sweet, multi-use topping in pastry. Typically made with just Flour, Butter and Sugar, it is not unlike a crumbled Shortbread Cookie. Streusel is quick and easy to make, and it can easily ...
Ingredients: ["butter", "sugar", "vanilla extract", "lemons", "flour", "cornmeal", "oats", "salt"]
Ingredients_raw: ["1   cup    butter (225g or 2 sticks)","1 1/4  cups    sugar (250g)","1   teaspoon    vanilla extract (5g)","2       lemons, zested ","2   cups    flour (240g)","3/4  cup    cornmeal (110g)","1   cup    oats (80g)","1   teaspoon    salt (6g)"]
Steps: ["Preheat the oven to 300 degrees Fahrenheit (149 degrees Celsius).", "Cream the Butter and Sugar in the bowl of an electric mixer fitted with a paddle attachment until smooth. Scrape down the bowl. A ...
Servings 1.0
Serving_size 1 (1101 g)
Tags 

In [15]:
import pandas as pd
from fuzzywuzzy import process
import gradio as gr

recipe = pd.read_csv("/content/recipes_ingredients.csv", on_bad_lines="skip", engine="python")


def find_recipe(query):

    choices = recipe['Name'].dropna().tolist()
    best_match, score = process.extractOne(query, choices)

    if score > 60:
        result = recipe[recipe['Name'] == best_match].iloc[0]
        return best_match, result["Ingredients"], result["Steps"],result["Description"],result["Ingredients_raw"],result["Servings"],result["Serving_size"],result["Tags"], score
    else:
        return None, None, None, None, None, None, None, None, None

print("RecipeBot: Hi! Ask me about any recipe (type 'exit' to quit)")

while True:
    query = input("\nYou: ")
    if query.lower() in ["exit", "quit", "bye"]:
        print("RecipeBot: Goodbye! Happy cooking! 🍲")
        break

    name, Ingredients, Steps, Description, Ingredients_raw, Servings, Serving_size, Tags, score = find_recipe(query)

    if Ingredients:
        print(f"\n RecipeBot: I think you meant *{name}* (match score: {score})")
        print("\n Ingredients:\n", Ingredients)
        print("\n Steps:\n", Steps[:400], "...")
        print("Description:",Description[:200],"...")
        print("Ingredients_raw:", Ingredients_raw)
        print("Servings",Servings)
        print("Serving_size",Serving_size)
        print("Tags", Tags[:200], "...")
    else:
        print(" RecipeBot: Sorry, I couldn't find a close match. Try another one!")


RecipeBot: Hi! Ask me about any recipe (type 'exit' to quit)

You: Peanut Butter and Banana Burrito

 RecipeBot: I think you meant *Peanut Butter and Banana Burrito* (match score: 100)

 Ingredients:
 ["flour tortilla", "peanut butter", "banana"]

 Steps:
 ["Heat tortilla over open flame of burner, till heated thru and pliable, turning once.", "Quickly spread peanut butter over 1/2 of tortilla, and top with banana slices.", "Fold in ends or tortilla, and then roll up!", "Enjoy!"] ...
Description: I know, I know...it's weird! But it's delicious! A great quick breakfast, lunch or snack!!! ...
Ingredients_raw: ["1       flour tortilla","1 -2   tablespoon    peanut butter","1/2      banana, sliced "]
Servings 1.0
Serving_size 1 (105 g)
Tags ["15-minutes-or-less", "time-to-make", "course", "main-ingredient", "cuisine", "preparation", "occasion", "north-american", "for-1-or-2", "5-ingredients-or-less", "breakfast", "lunch", "fruit", "ameri ...

You: exit
RecipeBot: Goodbye! Happy cooking! 🍲


In [16]:
import gradio as gr
import pandas as pd
import re

def chatbot(message, history):
    try:
        msg = str(message).lower()

        words = re.findall(r"\b\w+\b", msg)
        ingredients = [w for w in words if len(w) > 3]

        if not ingredients:
            return "Please mention at least one ingredient."

        mask = recipe["Ingredients"].astype(str).str.lower()
        matched = recipe[mask.apply(lambda x: any(ing in x for ing in ingredients))]

        if not matched.empty:
            first_recipe = matched.iloc[0]
            reply = f" You can make **{first_recipe['Name']}**!\n\n"
            reply += f" **Ingredients:**\n{first_recipe['Ingredients']}\n\n"
            reply += f" **Steps:**\n{first_recipe['Steps']}"
            return reply

        else:
            return f"⚠️ Sorry, I couldn't find any recipe with {', '.join(ingredients)}."

    except Exception as e:
        return f"⚠️ Internal Error: {str(e)}"

demo = gr.ChatInterface(
    fn=chatbot,
    title="Recipe Chatbot",
    description="What do you want to cook?"
)

demo.launch()


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://43a93b9eaac35dcc38.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
